# [M1] NÓI CHUYỆN VỚI DỮ LIỆU BẰNG PYTHON
## Buổi S02: Pandas Core: Đọc Dữ Liệu, DataFrame & Indexing/Filtering

---

### Metadata Khóa Học
* **Module:** M1 — NÓI CHUYỆN VỚI DỮ LIỆU BẰNG PYTHON
* **Buổi:** S02
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook truy vấn và lọc dữ liệu bảng chuyên nghiệp bằng Pandas
* **Mức độ liên quan VOAI / Olympic AI:** Khám phá cấu trúc đề thi, trích lọc dữ liệu nhanh không bị lỗi indexing.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu cấu trúc 2 tầng của Pandas: Series (1D có chỉ mục) và DataFrame (2D bảng dữ liệu).
- Đọc và kiểm tra nhanh chóng file dữ liệu CSV bằng `info()`, `describe()`, `head()`, `shape`.
- Phân biệt rạch ròi và làm chủ phương thức truy vấn vị trí `.iloc` vs nhãn `.loc`.
- Xây dựng các bộ lọc điều kiện phức tạp (Complex Conditional Filtering) bằng toán tử logic `&`, `|`, `~`.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M1_S01 (NumPy & Vectorization).
- Hiểu cơ bản về cấu trúc file bảng (CSV, hàng và cột).

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Dữ liệu đời thực và dữ liệu thi Olympic AI không bao giờ đến dưới dạng các ma trận số sạch sẽ. Chúng nằm trong các file bảng (CSV, Parquet, SQL) chứa hỗn hợp nhiều kiểu dữ liệu: tên thí sinh (string), độ tuổi (int), mức lương (float), thời gian nộp bài (datetime).

NumPy chỉ làm việc tốt với dữ liệu đồng nhất một kiểu. Để "nói chuyện" được với dữ liệu bảng phức tạp này, bạn bắt buộc phải sử dụng **Pandas**. Trong thi đấu AI, 70% thời gian của một chuyên gia dữ liệu là thao tác với Pandas để bóc tách, chọn lọc và biến đổi các dòng dữ liệu tiềm năng.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Series vs DataFrame trong Pandas
* **Pandas Series:** Là mảng một chiều có gán nhãn chỉ mục (Indexed 1D Array). Bạn có thể truy cập phần tử qua nhãn tên hoặc qua số thứ tự.
* **Pandas DataFrame:** Là bảng dữ liệu hai chiều có cấu trúc dạng bảng (2D Table), gồm nhiều Series có chung Index ghép lại với nhau. Mỗi cột là một Series đại diện cho một đặc trưng (Feature).

### 2. Hai quy tắc cốt tử khi truy xuất: `.loc` vs `.iloc`
* **`.loc[row_label, col_label]` (Label-based):** Truy xuất dựa trên **Tên nhãn** của hàng và cột.
  - Ví dụ: `df.loc[0:5, ['Age', 'Score']]` lấy các dòng có index mang tên từ 0 đến 5 (bao gồm cả 5) và các cột có tên 'Age', 'Score'.
* **`.iloc[row_pos, col_pos]` (Integer position-based):** Truy xuất dựa trên **Vị trí chỉ số nguyên** (từ 0 đến $N-1$), tuân theo quy tắc cắt lát chuẩn của Python (không bao gồm cận cuối).
  - Ví dụ: `df.iloc[0:5, 1:3]` lấy đúng 5 dòng đầu tiên (0, 1, 2, 3, 4) và 2 cột ở vị trí 1, 2.

### 3. Lọc dữ liệu theo điều kiện (Boolean Indexing)
Trong Pandas, ta dùng toán tử bitwise:
- `&` thay cho `and`
- `|` thay cho `or`
- `~` thay cho `not`
*Bắt buộc phải bọc từng biểu thức điều kiện trong dấu ngoặc đơn `()` để đảm bảo thứ tự ưu tiên toán tử!*



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Cấu trúc Pandas DataFrame:
                 Columns: ['Name', 'Age', 'Score']
                    │        │       │
      Index (Row)   ▼        ▼       ▼
           0     ['An',     19,     88.5]
           1     ['Bình',   20,     92.0]  <─── Row 1 (df.iloc[1])
           2     ['Cúc',    18,     79.0]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import pandas as pd
import numpy as np

# 1. Tao tap du lieu mau mo phong ho so thi sinh thi Olympic AI
np.random.seed(42)
n_samples = 10

data = {
    'Student_ID': [f'VOAI_{i:03d}' for i in range(1, n_samples + 1)],
    'Name': ['An', 'Binh', 'Chi', 'Dung', 'Em', 'Giang', 'Hoa', 'Khoa', 'Lan', 'Minh'],
    'Region': ['North', 'South', 'North', 'Central', 'South', 'North', 'Central', 'South', 'North', 'Central'],
    'Age': np.random.randint(18, 22, size=n_samples),
    'Math_Score': np.random.uniform(6.0, 10.0, size=n_samples).round(1),
    'Code_Score': np.random.uniform(5.0, 9.8, size=n_samples).round(1),
    'Passed_Round1': np.random.choice([True, False], size=n_samples, p=[0.7, 0.3])
}

df = pd.DataFrame(data)
print("===== 5 DONG DAU TIEN CUA DATAFRAME =====")
display_df = df.head()
print(display_df)

print("\n===== THONG TIN CAU TRUC DU LIEU (INFO) =====")
df.info()

### 🔍 Code Walkthrough (Khối 1)
Khối code tạo lập một DataFrame mẫu gồm 10 thí sinh với đa dạng kiểu dữ liệu (`object`, `int64`, `float64`, `bool`).
- Dùng `head()` để xem nhanh dữ liệu đại diện.
- Dùng `info()` để kiểm tra số lượng dòng, số cột non-null và kiểu dữ liệu (Dtype) của từng cột.
- **Tại sao viết như vậy:** Thói quen đầu tiên khi nhận bất kỳ dataset nào là phải chạy `head()` và `info()` để nắm cấu trúc tổng thể.


In [ ]:
# 2. Thao tac truy van loc du lieu: .loc, .iloc va Boolean Indexing

# Truong hop 1: Dung .iloc de lay 3 dong dau tien va 2 cot: Student_ID, Name (vi tri 0 va 1)
subset_iloc = df.iloc[0:3, 0:2]
print("--- Truy van bang .iloc (3 dong dau, cot 0-1) ---")
print(subset_iloc)

# Truong hop 2: Loc cac thi sinh o mien Bac ('North') VA co Math_Score >= 8.0
filter_cond = (df['Region'] == 'North') & (df['Math_Score'] >= 8.0)
top_north_students = df.loc[filter_cond, ['Student_ID', 'Name', 'Math_Score', 'Code_Score']]

print("\n--- Thi sinh mien Bac co diem Toan >= 8.0 (Boolean Indexing + .loc) ---")
print(top_north_students)

# Truong hop 3: Tinh cot moi: Tong diem (Total_Score) bang Vectorization trong Pandas
df['Total_Score'] = df['Math_Score'] + df['Code_Score']
print("\n--- Bang du lieu sau khi them cot Total_Score ---")
print(df[['Student_ID', 'Name', 'Total_Score']].head())

### 🔍 Code Walkthrough (Khối 2)
Khối code thực thi truy vấn chi tiết:
- Phân biệt rõ việc trích xuất bằng chỉ số vị trí `.iloc` và trích xuất bằng điều kiện kèm nhãn `.loc`.
- Thể hiện cú pháp lọc chuẩn mực với toán tử `&` và dấu ngoặc tròn `()`.
- Thao tác thêm một cột đặc trưng mới `Total_Score` được tính toán song song tức thì.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Đọc và kiểm tra tóm tắt thống kê
Chạy lệnh `df.describe()` trên DataFrame vừa tạo. Nêu rõ ý nghĩa của các dòng: `mean`, `std`, `50%` (trung vị).

### Bài 2 (Easy): Lọc theo điều kiện chuỗi
Lọc ra tất cả các thí sinh thuộc khu vực miền Nam (`'South'`) HOẶC có điểm lập trình (`Code_Score`) lớn hơn 8.5.

### Bài 3 (Medium): Truy vấn bằng phương thức `.query()`
Viết lại câu truy vấn ở Bài 2 bằng cách sử dụng cú pháp phương thức chuỗi `df.query(...)`. So sánh ưu nhược điểm về độ dễ đọc của `.query()` so với Boolean Masking truyền thống.

### Bài 4 (Medium): Sắp xếp thứ hạng (Sorting)
Sắp xếp bảng dữ liệu theo thứ tự giảm dần của `Total_Score`. Nếu điểm tổng bằng nhau, ưu tiên bạn có `Math_Score` cao hơn. Đặt lại chỉ mục (Reset Index) sau khi sắp xếp.

### Bài 5 (Challenge): Tạo cờ phân loại (Flag Feature)
Tạo thêm một cột mới tên là `'Grade'` trong DataFrame với quy tắc:
- `'Xuất sắc'` nếu `Total_Score >= 16.5`
- `'Khá'` nếu `14.0 <= Total_Score < 16.5`
- `'Trung bình'` nếu `Total_Score < 14.0`
*Yêu cầu:* Thực hiện bằng cách sử dụng `pd.cut()` hoặc hàm custom kết hợp `.apply()`.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Đề thi Olympic AI thường cung cấp file `train.csv` và `test.csv`. Một sai lầm phổ biến của thí sinh là dùng vòng lặp duyệt từng dòng để tính toán hoặc lọc dữ liệu, khiến notebook chạy mất 10 phút chỉ cho bước đọc dữ liệu ban đầu. Thành thạo `.loc`, `.iloc` và Boolean Filtering giúp bạn chỉ mất 2 dòng code để trích xuất tập dữ liệu con cần thiết trong chưa đầy 1 giây.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Một sinh viên viết điều kiện lọc dữ liệu và gặp lỗi `ValueError: The truth value of a Series is ambiguous`:

```python
# CODE CÓ LỖI SYNTAX LOGIC TRONG PANDAS:
# Thi sinh muon loc hoc vien co Age > 18 va Math_Score > 7.0
# bug_query = df[df['Age'] > 18 and df['Math_Score'] > 7.0] # Bao loi ngay lap tuc!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao từ khóa `and` của Python thuần gây ra lỗi trên khi áp dụng cho hai Pandas Series.
2. Viết lại biểu thức lọc chuẩn xác bằng toán tử bitwise và cặp ngoặc đơn.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Sự khác biệt căn bản giữa `.loc` và `.iloc` trong Pandas là gì?
   - A. `.loc` chạy nhanh hơn `.iloc` gấp 10 lần.
   - B. `.loc` dựa trên nhãn tên (Labels), còn `.iloc` dựa trên vị trí chỉ số nguyên (Integer positions).
   - C. `.iloc` chỉ dùng được cho cột, không dùng được cho dòng.
   - D. Không có sự khác biệt nào.

2. Cú pháp nào sau đây là ĐÚNG khi lọc các dòng có cột `'A' > 10` và cột `'B' < 5`?
   - A. `df[df['A'] > 10 and df['B'] < 5]`
   - B. `df[(df['A'] > 10) & (df['B'] < 5)]`
   - C. `df[(df['A'] > 10) && (df['B'] < 5)]`
   - D. `df[df['A'] > 10 , df['B'] < 5]`

3. Phương thức `df.info()` KHÔNG cung cấp thông tin nào sau đây?
   - A. Tổng số lượng hàng và cột.
   - B. Tên và kiểu dữ liệu (Dtype) của từng cột.
   - C. Số lượng giá trị không bị khuyết (Non-null count).
   - D. Giá trị dự đoán của mô hình AI.

4. Đoạn code `df.iloc[0:3]` sẽ trả về bao nhiêu hàng?
   - A. 2 hàng (hàng 0 và 1)
   - B. 3 hàng (hàng 0, 1 và 2)
   - C. 4 hàng (hàng 0, 1, 2 và 3)
   - D. Báo lỗi Indexing

5. Khi đọc một file dữ liệu lớn từ đĩa, phương thức nào của Pandas được sử dụng phổ biến nhất?
   - A. `pd.read_file()`
   - B. `pd.open_csv()`
   - C. `pd.read_csv()`
   - D. `pd.load_table()`



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Nắm vững cấu trúc Series và DataFrame.
  - Sử dụng thành thạo `info()`, `describe()`, `head()` để kiểm tra dataset.
  - Phân biệt tuyệt đối `.loc` và `.iloc`.
  - Thực thi Boolean Masking với các toán tử logic `&`, `|`, `~`.
* **What I can now do:**
  - Tải file bảng dữ liệu và bóc tách các tập dữ liệu con theo yêu cầu nghiệp vụ.
  - Tạo các cột đặc trưng mới một cách an toàn và nhanh chóng.
* **What I still need to practice:**
  - Học cách đối mặt với dữ liệu bẩn: Missing values, sai kiểu dữ liệu và trùng lặp ở Buổi 3.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M1_S03**, chúng ta sẽ tấn công vào bài toán thực tế nhất của Data Science: **Làm sạch dữ liệu (Data Cleaning)** — Xử lý các giá trị khuyết (Missing Values / NaNs), ép đúng kiểu dữ liệu (Data Types Casting), và loại bỏ trùng lặp (Duplicates) để bảo toàn tính trung thực của dữ liệu thi đấu!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M1**:
- Đọc dữ liệu CSV & kiểm tra DataFrame ✓
- Cấu trúc Series & DataFrame ✓
- Indexing & Filtering bằng .loc, .iloc và Boolean conditions ✓

* **Session Deliverable:** Mã nguồn truy vấn, trích xuất và lọc tập con dữ liệu chuẩn xác
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
